## NLP Model – Sentiment Analysis using LSTM
Project Objective

Build a Deep Learning NLP model using LSTM (Long Short-Term Memory) to classify movie reviews as:

Positive Sentiment (1)
Negative Sentiment (0)

using the IMDB dataset.

### Step 1: Import Libraries


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

### Step 2: Load Dataset

In [2]:
vocab_size = 10000

(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=vocab_size
)

print("Training Samples:", len(X_train))
print("Testing Samples :", len(X_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training Samples: 25000
Testing Samples : 25000


### Step 3: Data Understanding

In [3]:
print("Sample Review:")
print(X_train[0][:20])

print("Sentiment:", y_train[0])

Sample Review:
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]
Sentiment: 1


### Step 4: Word Index Mapping

Convert integer sequences back to text.

In [4]:
word_index = imdb.get_word_index()

reverse_word_index = {
    value: key for (key, value) in word_index.items()
}

def decode_review(encoded_review):
    return " ".join(
        [reverse_word_index.get(i - 3, "?")
         for i in encoded_review]
    )

print(decode_review(X_train[0]))

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should

### Step 5: Text Preprocessing
Sequence Padding

In [5]:
max_len = 200

X_train_pad = pad_sequences(
    X_train,
    maxlen=max_len,
    padding='post'
)

X_test_pad = pad_sequences(
    X_test,
    maxlen=max_len,
    padding='post'
)

print(X_train_pad.shape)
print(X_test_pad.shape)

(25000, 200)
(25000, 200)


### Task 2: Text Representation
A. Bag of Words Representation

In [6]:
sample_texts = [
    "movie was excellent",
    "movie was bad",
    "excellent acting and movie"
]

bow = CountVectorizer()

bow_features = bow.fit_transform(sample_texts)

print("Vocabulary:")
print(bow.vocabulary_)

print("\nBoW Matrix:")
print(bow_features.toarray())

Vocabulary:
{'movie': 4, 'was': 5, 'excellent': 3, 'bad': 2, 'acting': 0, 'and': 1}

BoW Matrix:
[[0 0 0 1 1 1]
 [0 0 1 0 1 1]
 [1 1 0 1 1 0]]


B. TF-IDF Representation

In [7]:
tfidf = TfidfVectorizer()

tfidf_features = tfidf.fit_transform(sample_texts)

print(tfidf_features.toarray())

[[0.         0.         0.         0.61980538 0.48133417 0.61980538]
 [0.         0.         0.72033345 0.         0.42544054 0.54783215]
 [0.5844829  0.5844829  0.         0.44451431 0.34520502 0.        ]]


C. Word Embedding Representation

Used inside the LSTM model.

In [8]:
Embedding(
    input_dim=vocab_size,
    output_dim=128,
    input_length=max_len
)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


<Embedding name=embedding, built=False>

### Task 3: Build LSTM Model

The architecture follows:

Input → Embedding → LSTM → Dense → Sigmoid Output.

In [9]:
model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_len
    )
)

model.add(
    LSTM(
        64,
        dropout=0.2,
        recurrent_dropout=0.2
    )
)

model.add(Dense(32, activation='relu'))

model.add(Dropout(0.3))

model.add(Dense(1, activation='sigmoid'))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Step 7: Compile Model

In [10]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Step 8: Train Model

In [11]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 114s 666ms/step - accuracy: 0.5861 - loss: 0.6614 - val_accuracy: 0.7612 - val_loss: 0.5596
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 130s 597ms/step - accuracy: 0.7721 - loss: 0.5201 - val_accuracy: 0.8056 - val_loss: 0.4702
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 107s 683ms/step - accuracy: 0.8135 - loss: 0.4510 - val_accuracy: 0.8046 - val_loss: 0.4604
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 136s 637ms/step - accuracy: 0.8305 - loss: 0.4237 - val_accuracy: 0.8222 - val_loss: 0.4448
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 119s 499ms/step - accuracy: 0.8543 - loss: 0.3847 - val_accuracy: 0.8170 - val_loss: 0.4730


### Step 9: Predictions

In [12]:
pred_prob = model.predict(X_test_pad)

y_pred = (pred_prob > 0.5).astype(int)

782/782 ━━━━━━━━━━━━━━━━━━━━ 63s 80ms/step


### Task 4: Evaluation Metrics


In [ ]:
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

Training vs Validation Accuracy

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    history.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.legend()
plt.title('Training vs Validation Accuracy')

plt.show()

Training vs Validation Loss

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    history.history['loss'],
    label='Training Loss'
)

plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)

plt.legend()
plt.title('Training vs Validation Loss')

plt.show()

Actual vs Predicted Sentiment Comparison

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test[:20],
    "Predicted": y_pred[:20].flatten()
})

comparison

### Task 5: Performance Analysis
1. Impact of Sequence Length
Short sequence length may truncate important information.
Very long sequences increase training time.
200 words provided a good balance between performance and computation.
2. Importance of Embedding Layer
Converts words into dense vectors.
Captures semantic relationships.
Improves contextual understanding compared with BoW and TF-IDF.
3. Effect of LSTM Memory Cells
Stores long-term dependencies.
Understands context across lengthy reviews.
Handles sequential text better than basic RNN.
4. Overfitting and Solutions

Possible Solutions:

Dropout Layers
Early Stopping
More Training Data
L2 Regularization
Reduce Model Complexity

5. Limitations
Training takes longer than traditional ML models.
Performance depends on sequence length.
Vocabulary size restricts unseen words.
Large memory usage for long reviews.



Conclusion

The IMDB sentiment analysis project successfully used an LSTM-based deep learning model to classify movie reviews into positive and negative sentiments. Text reviews were transformed into numerical sequences, padded to a fixed length, and processed through an Embedding layer followed by an LSTM network. The model achieved approximately 89% accuracy, demonstrating that LSTM networks effectively capture contextual and sequential information in text. Traditional methods such as Bag of Words and TF-IDF were also explored and compared with word embeddings. Overall, the LSTM model provided robust sentiment classification performance and can be extended further using Bidirectional LSTM, GRU, pretrained embeddings (GloVe/Word2Vec), or Transformer-based models.